In [1]:
import pandas as pd 
import numpy as np 


In [2]:
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

print("Data loaded successfully!")

In [3]:
df = pd.merge(orders, items, on='order_id', how='inner')
df = pd.merge(df, products, on='product_id', how='inner')
df = pd.merge(df, customers, on='customer_id', how='inner')

print(df.head(3))

In [4]:
df = df[df['order_status'] == 'delivered']
df = df.dropna()
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['purchase_week'] = df['order_purchase_timestamp'].dt.to_period('W')

print("Data cleaned! Current shape:", df.shape)

In [5]:

summary = df.groupby(['customer_state', 'product_category_name']).agg({
    'price': 'sum',          
    'order_item_id': 'count' 
}).reset_index() 


summary = summary.rename(columns={
    'price': 'total_revenue',
    'order_item_id': 'items_sold'
})

summary = summary.sort_values(by='total_revenue', ascending=False)
print(summary.head(10))

In [8]:
# 1. Load the translation file (this should be in your downloaded Kaggle folder)
translations = pd.read_csv('product_category_name_translation.csv')

# 2. Merge the English translations into your summary table
# We use a 'left' join so we don't lose any rows if a translation is missing
final_report = pd.merge(summary, translations, on='product_category_name', how='left')

# 3. Reorder the columns so the English name sits right next to the state
final_report = final_report[['customer_state', 'product_category_name_english', 'total_revenue', 'items_sold']]

# 4. Save your hard work to a brand new CSV file!
# index=False prevents pandas from exporting those random row numbers (1265, 1263, etc.)
final_report.to_csv('brazil_ecommerce_summary.csv', index=False)

# Let's peek at the final, translated English version
print("Report successfully translated and saved!")
print(final_report.head(5))

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Grab just the top 5 categories from your final report
top_5 = final_report.head(5)

# 2. Set up the visual styling (Dark mode, terminal font)
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'

# 3. Create the canvas (10 inches wide, 6 inches tall)
plt.figure(figsize=(10, 6))

# 4. Build the bar chart
# We use a horizontal bar chart (y=category, x=revenue) because category names are long
ax = sns.barplot(
    x='total_revenue', 
    y='product_category_name_english', 
    data=top_5,
    color='#00ff41'  # A sharp, matrix-style neon green
)

# 5. Add a subtle grid layout to make the values easy to read
ax.grid(color='#333333', linestyle='--', linewidth=0.5, axis='x')

# 6. Add labels and a clean title
plt.title('>> TOP 5 REVENUE DRIVERS - SÃO PAULO', fontsize=14, weight='bold', pad=20)
plt.xlabel('TOTAL REVENUE (BRL)', fontsize=12)
plt.ylabel('PRODUCT CATEGORY', fontsize=12)

# 7. Clean up the edges by removing the top and right borders (despining)
sns.despine(left=True, bottom=True)

# 8. Render the plot!
plt.show()

In [8]:
import pandas as pd
import numpy as np

df_english = pd.merge(df, translations, on='product_category_name', how='left')

weekly_sales = df_english.groupby(['product_category_name_english', 'purchase_week']).agg({
    'order_item_id': 'count',  
    'price': 'mean'            
}).reset_index()

weekly_sales = weekly_sales.rename(columns={'order_item_id': 'units_sold', 'price': 'avg_price'})


volatility_model = weekly_sales.groupby('product_category_name_english').agg({
    'units_sold': ['mean', 'std'],  
    'avg_price': 'mean'             
}).reset_index()

volatility_model.columns = ['category', 'avg_weekly_demand', 'demand_volatility', 'avg_price']



volatility_model['holding_cost_per_unit'] = volatility_model['avg_price'] * 0.02

volatility_model['stockout_cost_per_unit'] = volatility_model['avg_price'] * 0.30



volatility_model = volatility_model.dropna().sort_values(by='demand_volatility', ascending=False)


print(volatility_model.head(5))

In [9]:
from scipy.stats import norm
import numpy as np

volatility_model['critical_ratio'] = volatility_model['stockout_cost_per_unit'] / (volatility_model['stockout_cost_per_unit'] + volatility_model['holding_cost_per_unit'])


volatility_model['z_score'] = norm.ppf(volatility_model['critical_ratio'])

volatility_model['safety_stock'] = np.ceil(volatility_model['z_score'] * volatility_model['demand_volatility'])

volatility_model['optimal_inventory'] = np.ceil(volatility_model['avg_weekly_demand'] + volatility_model['safety_stock'])

final_risk_model = volatility_model[[
    'category', 'avg_weekly_demand', 'safety_stock', 'optimal_inventory', 'critical_ratio'
]]

print(">> FINANCIAL RISK MODEL COMPLETE")
print(final_risk_model.head(5))

In [10]:
import matplotlib.pyplot as plt

# 1. Grab the top 5 categories and set the category name as the index for easy plotting
top_5_risk = final_risk_model.head(5).set_index('category')

# 2. Keep the sleek tech-aesthetic
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'

# 3. Create a stacked horizontal bar chart
# We plot Average Demand (Green) and Safety Stock (Red) - together they equal Optimal Inventory
ax = top_5_risk[['avg_weekly_demand', 'safety_stock']].plot(
    kind='barh', 
    stacked=True, 
    figsize=(11, 6),
    color=['#00ff41', '#ff0055'] # Neon green for baseline, neon pink for the risk buffer
)

# 4. Add the grid for readability
ax.grid(color='#333333', linestyle='--', linewidth=0.5, axis='x')

# 5. Labels and Titles
plt.title('>> OPTIMAL INVENTORY: DEMAND VS. SAFETY BUFFER', fontsize=14, weight='bold', pad=20)
plt.xlabel('TOTAL UNITS REQUIRED', fontsize=12)
plt.ylabel('PRODUCT CATEGORY', fontsize=12)

# 6. Clean up the legend and borders
plt.legend(['Average Demand', 'Safety Buffer'], loc='lower right', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 7. Render!
plt.show()

In [11]:
import numpy as np

# 1. Calculate weekly sales per WAREHOUSE (State) and Category
weekly_warehouse_sales = df_english.groupby(['customer_state', 'product_category_name_english', 'purchase_week']).agg({
    'order_item_id': 'count'
}).reset_index().rename(columns={'order_item_id': 'units_sold'})

# 2. Calculate Volatility per Warehouse
warehouse_model = weekly_warehouse_sales.groupby(['customer_state', 'product_category_name_english']).agg({
    'units_sold': ['mean', 'std']
}).reset_index()

# Flatten columns to make them easy to work with
warehouse_model.columns = ['warehouse_node', 'category', 'avg_demand', 'volatility']

# Drop categories that didn't have enough weeks to calculate a standard deviation
warehouse_model = warehouse_model.dropna()

# 3. Apply the Algorithmic Optimization 
# We will use the exact same Z-score probability (approx 1.53) from our 93.75% target in Phase 2
z_score = 1.53 

# Calculate the Dynamic Safety Stock for the specific warehouse
warehouse_model['dynamic_safety_stock'] = np.ceil(warehouse_model['volatility'] * z_score)

# THE ENGINE: Calculate the exact Reorder Point
warehouse_model['reorder_point'] = np.ceil(warehouse_model['avg_demand'] + warehouse_model['dynamic_safety_stock'])

# 4. Clean up and format the final output
final_algorithm = warehouse_model[['warehouse_node', 'category', 'avg_demand', 'dynamic_safety_stock', 'reorder_point']]

# Sort by the largest warehouse requirements
final_algorithm = final_algorithm.sort_values(by='reorder_point', ascending=False)

# Display the engine's top 10 recommendations!
print(">> ALGORITHMIC ENGINE: DYNAMIC REORDER POINTS GENERATED\n")
print(final_algorithm.head(10).to_string(index=False))